# Leaderboard improvement experiments

Reference: `outputs/v2/submission_3seed.csv`, identified by the user as the **0.86380** Kaggle submission, ranked second in the supplied screenshot. The screenshot shows a leading score of 0.86424 and an approximately 33% public test split. These are historical observations, not live leaderboard claims.

We preserve that CSV and its Gamma_GT predictions. This notebook records clinical-feature, label-adjusted urine plate, and complementary XGBoost classification experiments. Development selection excludes the earlier outer-check IDs; confirmation reuses development rows with another fold assignment. Full-data CV is diagnostic. A new CSV is promoted only after the gates in the scripts pass.

Run from the repository root using the `.venv` Python interpreter. Earlier V2 outputs and the v1 split manifest are required. XGBoost on macOS also requires the OpenMP runtime (`brew install libomp`). The retraining cell launches isolated Python processes because the experiment wrappers customize a shared workflow. No cells upload to Kaggle.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display, Code
ROOT = Path.cwd()
assert (ROOT / 'data/train.csv').exists(), 'Run from the project root after downloading the competition data.'
for filename in ['smoking_push.py', 'supervised_smoking.py', 'xgboost_push.py', 'ordered_smoking_push.py', 'gamma_bias_push.py']:
    print(filename)
    display(Code(filename=str(ROOT / filename), language='python'))

## Reproduce the experiments

The exact leave-one-out urine feature removes a row's own Smoking label from both the marker/Smoking coefficient and the plate offset. Tests compare the formula with physically excluding each row. XGBoost uses categorical mappings fitted on the training partition; unknown evaluation categories become missing values. Source: [official XGBoost categorical documentation](https://xgboost.readthedocs.io/en/stable/tutorials/categorical.html).

In [ ]:
RUN_RETRAIN = False
if RUN_RETRAIN:
    import subprocess
    for script in ['smoking_push.py', 'supervised_smoking.py', 'xgboost_push.py', 'ordered_smoking_push.py', 'gamma_bias_push.py']:
        subprocess.run([str(ROOT / '.venv/bin/python'), script, *([] if script == 'gamma_bias_push.py' else ['--phase', 'all'])], check=True, cwd=ROOT)
else:
    print('Displaying saved results. Enable RUN_RETRAIN to reproduce the experiments.')

In [ ]:
directories = [ROOT / 'outputs' / name for name in ['smoking_push_consistent', 'supervised_smoking_push', 'xgboost_smoking_push', 'ordered_smoking_push', 'gamma_bias_push']]
for directory in directories:
    print(directory.name)
    for filename in ['selection.csv', 'confirmation.json', 'final_cv.json']:
        path = directory / filename
        if path.exists():
            if path.suffix == '.csv':
                table = pd.read_csv(path)
                key = 'auc' if 'auc' in table else 'score_gain'
                display(table.sort_values(key, ascending=False))
            else:
                display(json.loads(path.read_text()))

## Verify any promoted CSV

Existing submitted predictions are protected. A promoted Smoking candidate retains the successful Gamma_GT values; a Gamma_GT correction retains Smoking values. Passing a local gate does not guarantee a higher public or private Kaggle score. No notebook cell submits to Kaggle.

In [ ]:
reference = pd.read_csv(ROOT / 'outputs/v2/submission_3seed.csv')
sample = pd.read_csv(ROOT / 'data/sample_submission.csv')
assert reference.ID.equals(sample.ID)
for directory in directories:
    path = directory / ('submission_gamma_bias.csv' if directory.name == 'gamma_bias_push' else 'submission_smoking_push.csv')
    if path.exists():
        candidate = pd.read_csv(path)
        assert candidate.ID.equals(sample.ID) and len(candidate) == 6399
        assert candidate.columns.tolist() == ['ID', 'Smoking', 'Gamma_GT']
        assert np.isfinite(candidate[['Smoking', 'Gamma_GT']]).all().all()
        assert candidate.Smoking.between(0, 1).all()
        assert candidate.Gamma_GT.between(1, 1000).all()
        if directory.name == 'gamma_bias_push':
            np.testing.assert_array_equal(candidate.Smoking, reference.Smoking)
        else:
            np.testing.assert_array_equal(candidate.Gamma_GT, reference.Gamma_GT)
        print('Validated:', path)
    else:
        print(directory.name, ': no promoted submission.')

## Result of this round

No experiment passed all promotion checks. The supervised cotinine blend gained only 0.00008850 AUC in the final diagnostic, below the predefined 0.0002 gate. The ordered CatBoost blend lost 0.00002859 AUC. Keep `outputs/v2/submission_3seed.csv`, the confirmed 0.86380 Kaggle submission, as the reference. Its file hash was checked unchanged. No new Kaggle submission was made.

The next priority is fully nested validation of the inherited Gamma_GT calibration and regression blend, followed by focused error analysis for floor readings, missing rapid tests, and high Gamma_GT values.